# 🧪 Synthetic Telecom Customer Dataset Generation

## Objective

The original IBM Telco Customer Churn dataset contains approximately 7,000 customers.

To simulate a production-scale analytics environment, this project generates a larger synthetic dataset while preserving important statistical characteristics and business relationships observed in the original dataset.

The synthetic dataset will contain 500,000 customer records.

The generated data will be used to demonstrate:

- Scalable data processing
- ETL pipeline development
- PostgreSQL data warehousing
- SQL business analysis
- Power BI reporting

> Note: The 500,000-row dataset is synthetically generated for scalability and pipeline demonstration. It is not presented as real telecom customer data.

## Load our feature-engineered dataset

In [1]:
import pandas as pd
import numpy as np

# Load feature-engineered dataset
df = pd.read_csv(
    "../data/processed/telco_customer_churn_features.csv"
)

print("Dataset shape:", df.shape)

df.head()

Dataset shape: (7032, 31)


,customer_id,gender,senior_citizen,partner,dependents,tenure,phone_service,multiple_lines,internet_service,online_security,...,churn_flag,tenure_segment,monthly_charge_segment,estimated_customer_value,service_count,customer_value_segment,new_customer_flag,high_monthly_charge_flag,risk_score,risk_segment
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,0,0-6 Months,Low,29.85,1,Low Value,1,0,2,High Risk
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,0,25-48 Months,Medium,1936.30,2,High Value,0,0,0,Low Risk
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,1,0-6 Months,Medium,107.70,2,Low Value,1,0,2,High Risk
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,0,25-48 Months,Medium,1903.50,3,High Value,0,0,0,Low Risk
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,1,0-6 Months,High,141.40,0,Low Value,1,0,2,High Risk


### Understand the distributions

In [2]:
categorical_columns = [
    "gender",
    "senior_citizen",
    "partner",
    "dependents",
    "phone_service",
    "multiple_lines",
    "internet_service",
    "online_security",
    "online_backup",
    "device_protection",
    "tech_support",
    "streaming_tv",
    "streaming_movies",
    "contract",
    "paperless_billing",
    "payment_method",
    "churn"
]

for col in categorical_columns:
    print(f"\n{'='*50}")
    print(col)
    print(df[col].value_counts(normalize=True).round(3))


gender
gender
Male      0.505
Female    0.495
Name: proportion, dtype: float64

senior_citizen
senior_citizen
0    0.838
1    0.162
Name: proportion, dtype: float64

partner
partner
No     0.517
Yes    0.483
Name: proportion, dtype: float64

dependents
dependents
No     0.702
Yes    0.298
Name: proportion, dtype: float64

phone_service
phone_service
Yes    0.903
No     0.097
Name: proportion, dtype: float64

multiple_lines
multiple_lines
No                  0.481
Yes                 0.422
No phone service    0.097
Name: proportion, dtype: float64

internet_service
internet_service
Fiber optic    0.440
DSL            0.344
No             0.216
Name: proportion, dtype: float64

online_security
online_security
No                     0.497
Yes                    0.287
No internet service    0.216
Name: proportion, dtype: float64

online_backup
online_backup
No                     0.439
Yes                    0.345
No internet service    0.216
Name: proportion, dtype: float64

device_prote

## Create probability distributions

In [3]:
def get_probabilities(data, column):
    return data[column].value_counts(normalize=True)

In [4]:
contract_prob = get_probabilities(df, "contract")

contract_prob

contract
Month-to-month    0.551052
Two year          0.239619
One year          0.209329
Name: proportion, dtype: float64

## Understand numeric distributions

In [5]:
numeric_columns = [
    "tenure",
    "monthly_charges",
    "total_charges"
]

df[numeric_columns].describe()

,tenure,monthly_charges,total_charges
count,7032.000000,7032.000000,7032.000000
mean,32.421786,64.798208,2283.300441
std,24.545260,30.085974,2266.771362
min,1.000000,18.250000,18.800000
25%,9.000000,35.587500,401.450000
50%,29.000000,70.350000,1397.475000
75%,55.000000,89.862500,3794.737500
max,72.000000,118.750000,8684.800000


## Create the generator

In [6]:
def generate_categorical_column(data, column, n):
    probabilities = data[column].value_counts(normalize=True)

    return np.random.choice(
        probabilities.index,
        size=n,
        p=probabilities.values
    )

## Set dataset size

In [7]:
N = 500_000

np.random.seed(42)

print("Generating", N, "customers...")

Generating 500000 customers...


## Generate customer IDs

In [8]:
synthetic = pd.DataFrame()

synthetic["customer_id"] = [
    f"SYN{str(i).zfill(7)}"
    for i in range(1, N + 1)
]

synthetic.head()

,customer_id
0,SYN0000001
1,SYN0000002
2,SYN0000003
3,SYN0000004
4,SYN0000005


## Generate categorical attributes

In [9]:
for col in categorical_columns:
    if col != "churn":
        synthetic[col] = generate_categorical_column(
            df,
            col,
            N
        )

In [10]:
synthetic.head()

,customer_id,gender,senior_citizen,partner,dependents,phone_service,multiple_lines,internet_service,online_security,online_backup,device_protection,tech_support,streaming_tv,streaming_movies,contract,paperless_billing,payment_method
0,SYN0000001,Male,0,Yes,Yes,Yes,No phone service,DSL,Yes,No internet service,No,Yes,No internet service,No,One year,Yes,Mailed check
1,SYN0000002,Female,0,No,No,Yes,No,Fiber optic,No,No,No,No internet service,No,No internet service,Month-to-month,Yes,Bank transfer (automatic)
2,SYN0000003,Female,0,No,No,Yes,No,DSL,Yes,No,No internet service,No,No internet service,No,Month-to-month,Yes,Credit card (automatic)
3,SYN0000004,Female,1,Yes,No,Yes,No,Fiber optic,Yes,No,No,Yes,Yes,Yes,Month-to-month,Yes,Mailed check
4,SYN0000005,Male,0,Yes,No,Yes,No,No,Yes,No internet service,Yes,No,No internet service,No internet service,Month-to-month,Yes,Credit card (automatic)


## Generate Tenure

In [11]:
synthetic["tenure"] = np.random.choice(
    df["tenure"].values,
    size=N,
    replace=True
)

## Generate Monthly Charges

In [12]:
synthetic["monthly_charges"] = np.random.choice(
    df["monthly_charges"].values,
    size=N,
    replace=True
)

synthetic["monthly_charges"] = (
    synthetic["monthly_charges"]
    .round(2)
)

## Generate Total Charges

In [13]:
noise = np.random.normal(
    loc=0,
    scale=50,
    size=N
)

synthetic["total_charges"] = (
    synthetic["monthly_charges"]
    * synthetic["tenure"]
    + noise
)

In [14]:
synthetic["total_charges"] = (
    synthetic["total_charges"]
    .clip(lower=0)
    .round(2)
)

## Create a more realistic churn mechanism 

In [15]:
churn_probability = np.full(N, 0.20)

In [ ]:
# Contract effect
churn_probability += np.where(
    synthetic["contract"] == "Month-to-month",
    0.15,
    0
)

churn_probability += np.where(
    synthetic["contract"] == "Two year",
    -0.10,
    0
)

In [17]:
# Tenure effect
churn_probability += np.where(
    synthetic["tenure"] <= 6,
    0.10,
    0
)

churn_probability += np.where(
    synthetic["tenure"] >= 48,
    -0.08,
    0
)

In [18]:
# Monthly charge effect
charge_75th = df["monthly_charges"].quantile(0.75)

churn_probability += np.where(
    synthetic["monthly_charges"] >= charge_75th,
    0.05,
    0
)

In [19]:
# Internet service effect
churn_probability += np.where(
    synthetic["internet_service"] == "Fiber optic",
    0.05,
    0
)

In [20]:
# Payment method effect
churn_probability += np.where(
    synthetic["payment_method"] == "Electronic check",
    0.05,
    0
)

In [21]:
churn_probability = np.clip(
    churn_probability,
    0.02,
    0.75
)

In [22]:
# Generate Churn
synthetic["churn"] = np.where(
    np.random.random(N) < churn_probability,
    "Yes",
    "No"
)

In [23]:
synthetic["churn"].value_counts(normalize=True)

churn
No     0.695436
Yes    0.304564
Name: proportion, dtype: float64

## Generate analytical features

In [24]:
# Churn flag
synthetic["churn_flag"] = synthetic["churn"].map({
    "Yes": 1,
    "No": 0
})

In [25]:
# Tenure segment
synthetic["tenure_segment"] = pd.cut(
    synthetic["tenure"],
    bins=[-1, 6, 12, 24, 48, 60, 72],
    labels=[
        "0-6 Months",
        "7-12 Months",
        "13-24 Months",
        "25-48 Months",
        "49-60 Months",
        "61-72 Months"
    ]
)

In [26]:
# Monthly charge segment
synthetic["monthly_charge_segment"] = pd.cut(
    synthetic["monthly_charges"],
    bins=[0, 30, 60, 90, 120, np.inf],
    labels=[
        "Low",
        "Medium",
        "High",
        "Very High",
        "Premium"
    ]
)

In [27]:
# Estimated customer value
synthetic["estimated_customer_value"] = (
    synthetic["monthly_charges"]
    * synthetic["tenure"]
).round(2)

In [28]:
# Service count
service_columns = [
    "online_security",
    "online_backup",
    "device_protection",
    "tech_support",
    "streaming_tv",
    "streaming_movies"
]

synthetic["service_count"] = (
    synthetic[service_columns]
    .eq("Yes")
    .sum(axis=1)
)

In [29]:
# New customer flag
synthetic["new_customer_flag"] = np.where(
    synthetic["tenure"] <= 6,
    1,
    0
)

In [30]:
# High monthly charge flag
synthetic["high_monthly_charge_flag"] = np.where(
    synthetic["monthly_charges"] >= charge_75th,
    1,
    0
)

In [31]:
# Risk score
synthetic["risk_score"] = (
    (synthetic["contract"] == "Month-to-month").astype(int)
    + synthetic["new_customer_flag"]
    + synthetic["high_monthly_charge_flag"]
)

In [32]:
synthetic["risk_segment"] = pd.cut(
    synthetic["risk_score"],
    bins=[-1, 0, 1, 3],
    labels=[
        "Low Risk",
        "Medium Risk",
        "High Risk"
    ]
)

In [33]:
# Validate the generated dataset
print("Shape:", synthetic.shape)

Shape: (500000, 30)


In [34]:
synthetic.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 30 columns):
 #   Column                    Non-Null Count   Dtype   
---  ------                    --------------   -----   
 0   customer_id               500000 non-null  object  
 1   gender                    500000 non-null  object  
 2   senior_citizen            500000 non-null  int64   
 3   partner                   500000 non-null  object  
 4   dependents                500000 non-null  object  
 5   phone_service             500000 non-null  object  
 6   multiple_lines            500000 non-null  object  
 7   internet_service          500000 non-null  object  
 8   online_security           500000 non-null  object  
 9   online_backup             500000 non-null  object  
 10  device_protection         500000 non-null  object  
 11  tech_support              500000 non-null  object  
 12  streaming_tv              500000 non-null  object  
 13  streaming_movies          500

In [35]:
print(
    "Duplicate customer IDs:",
    synthetic["customer_id"].duplicated().sum()
)

Duplicate customer IDs: 0


In [36]:
comparison = pd.DataFrame({
    "Original": df["churn"].value_counts(normalize=True),
    "Synthetic": synthetic["churn"].value_counts(normalize=True)
})

comparison

,Original,Synthetic
churn,,
No,0.734215,0.695436
Yes,0.265785,0.304564


In [37]:
contract_comparison = pd.DataFrame({
    "Original": df["contract"].value_counts(normalize=True),
    "Synthetic": synthetic["contract"].value_counts(normalize=True)
})

contract_comparison

,Original,Synthetic
contract,,
Month-to-month,0.551052,0.551454
Two year,0.239619,0.239414
One year,0.209329,0.209132


In [38]:
pd.crosstab(
    synthetic["contract"],
    synthetic["churn"],
    normalize="index"
).mul(100).round(2)

churn,No,Yes
contract,,
Month-to-month,60.50,39.50
One year,75.36,24.64
Two year,85.29,14.71


In [39]:
synthetic.groupby("churn")["tenure"].mean()

churn
No     34.459542
Yes    27.853023
Name: tenure, dtype: float64

In [40]:
synthetic.groupby("churn")["monthly_charges"].mean()

churn
No     64.172409
Yes    66.303008
Name: monthly_charges, dtype: float64

In [41]:
output_path = "../data/synthetic/telco_customer_churn_500k.csv"

synthetic.to_csv(
    output_path,
    index=False
)

print(f"✅ Synthetic dataset saved to: {output_path}")

✅ Synthetic dataset saved to: ../data/synthetic/telco_customer_churn_500k.csv
